> **Generated notebook — do not edit here.**  
> Source: `01_scripts/03_gene_functional.Rmd`, which is also a chapter of the course book.  
> To change anything, edit the Rmd and run `python3 util/rmd_to_ipynb.py`.
>
> Run the notebooks in order — **01 → 02 → 03** — with the **R** kernel; each step saves results that the next one loads.

In [ ]:
# Match the report's defaults: warnings hidden (warning=FALSE in the Rmd)
# and 7 x 5 inch figures. Remove the warn option to see warnings.
options(warn = -1, repr.plot.width = 7, repr.plot.height = 5)

# Gene Functional Enrichment Analysis

This report covers functional enrichment analysis of differentially expressed genes from primary human airway smooth muscle (ASM) cells treated with dexamethasone ([Himes *et al.*, 2014](https://doi.org/10.1371/journal.pone.0099625)).

We apply two complementary approaches:

- **ORA** (Over-Representation Analysis) using [`gprofiler2`](https://cran.r-project.org/package=gprofiler2) — tests whether significant DE genes are enriched in curated gene sets (GO, KEGG, Reactome) more than expected by chance. g:Profiler natively supports human, so no ID mapping workaround is needed.
- **GSEA** (Gene Set Enrichment Analysis) using [`fgsea`](https://bioconductor.org/packages/fgsea/) with MSigDB gene sets from [`msigdbr`](https://cran.r-project.org/package=msigdbr) — uses the full ranked gene list to detect coordinated pathway-level shifts, including subtle changes below the significance threshold.

### **Why two tools?**

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

 ORA (g:Profiler) answers "are my significant genes over-represented in known gene sets?" using a
 hypergeometric test against a background. GSEA (fgsea) uses the *ranked* full gene list and returns
 a Normalised Enrichment Score (NES), catching coordinated shifts that ORA misses. Using each tool for
 what it does best gives the most reliable picture.

</div>

### **Why these gene sets?**

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

For human data, GO Biological Process, KEGG and Reactome (via g:Profiler) are the standard, well-curated
choices and are directly interpretable. The original Himes 2014 study used GO-based enrichment. For GSEA we
use the MSigDB Reactome collection, a curated set of detailed pathways that includes the immune and
inflammatory processes relevant to this experiment.

</div>

### Load Libraries

In [ ]:
library(tidyverse)
library(gprofiler2)
library(fgsea)
library(msigdbr)
library(knitr)
library(kableExtra)
library(DT)

## Load Results from Differential Expression Analysis

This script reads the DE results produced by `02_differential_expression_analysis.Rmd`.
Make sure you have run that script first — it saves two files to `results/`:

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

- `results/DE_treatment_vs_control_all.tsv` — all tested genes with shrunken LFC
- `results/DE_treatment_vs_control_significant.tsv` — significant DE genes (padj < 0.05, |LFC| >= 1)

</div>

In [ ]:
git_root <- system("git rev-parse --show-toplevel", intern = TRUE)

res_df <- read.table(
  file.path(git_root, "results", "human", "DE_treatment_vs_control_all.tsv"),
  header      = TRUE,
  sep         = "\t",
  check.names = TRUE
)

res_sig <- read.table(
  file.path(git_root, "results", "human", "DE_treatment_vs_control_significant.tsv"),
  header      = TRUE,
  sep         = "\t",
  check.names = TRUE
)

cat("All tested genes     :", nrow(res_df), "\n")
cat("Significant DE genes :", nrow(res_sig), "\n")
cat("  Upregulated        :", sum(res_sig$log2FoldChange >= 1), "\n")
cat("  Downregulated      :", sum(res_sig$log2FoldChange <= -1), "\n")

<div style="background:#d1ecf1;border-left:4px solid #0c5460;padding:10px;margin:10px 0;">

  <strong>Note:</strong> The <code>gene</code> column holds human gene symbols (from the reference GTF).
  g:Profiler and MSigDB both accept gene symbols directly, so the gene names from the pipeline are used
  as-is with no ID conversion step.

</div>

## Part 1 — Over-Representation Analysis (ORA) with g:Profiler

**What is ORA?**

**ORA asks: *"Are my significant DE genes enriched in any gene set more than expected by chance?"***

g:Profiler runs a hypergeometric test for each gene set and applies its own multiple-testing correction (`g_SCS` by default, which accounts for the overlap structure of GO/pathway terms).

- **Query set** — significant DE genes (padj < 0.05, |LFC| >= 1), split by direction (up- and down-regulated)
- **Background (custom)** — all genes tested by DESeq2 (`domain_scope = "custom"`)
- **Gene sets** — GO Biological Process, KEGG, Reactome

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

**Key assumption:** ORA treats all significant genes equally — it ignores the magnitude of fold change.
Running ORA separately on up- and down-regulated genes partially addresses this, which is why we split
the query set below. For a fully ranked approach, see Part 2 (GSEA).

</div>

### Prepare Input

We split significant genes by direction. The background is all genes tested by DESeq2, regardless of direction.

In [ ]:
sig_up   <- res_sig %>% filter(log2FoldChange >= 1)  %>% pull(gene)
sig_down <- res_sig %>% filter(log2FoldChange <= -1) %>% pull(gene)
background <- res_df$gene

cat("Up-regulated genes      :", length(sig_up),     "\n")
cat("Down-regulated genes    :", length(sig_down),   "\n")
cat("Background (all tested) :", length(background), "\n")

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px;margin:10px 0;">

  <strong>Tip:</strong> Always use all <em>tested</em> genes as the background, not the whole genome.
  DESeq2 already filtered to expressed genes, so <code>res_df$gene</code> is the correct pool.
  Passing it via <code>custom_bg</code> with <code>domain_scope = "custom"</code> tells g:Profiler
  to use exactly this background.

</div>

### Run ORA

g:Profiler is queried once per direction. The result object's `$result` slot is a data frame; we keep the
GO:BP, KEGG and Reactome sources.

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px;margin:10px 0;">

  <strong>Tip:</strong> <code>gost()</code> makes a live API call and needs an internet connection.
  If you are offline, this chunk will not run.

</div>

In [ ]:
run_ora_gprofiler <- function(query, background, sources = c("GO:BP", "KEGG", "REAC")) {
  if (length(query) == 0) return(NULL)
  res <- gost(
    query                   = query,
    organism                = "hsapiens",
    ordered_query           = FALSE,
    significant             = TRUE,
    user_threshold          = 0.05,
    correction_method       = "g_SCS",
    domain_scope            = "custom",
    custom_bg               = background,
    sources                 = sources,
    evcodes                 = FALSE
  )
  if (is.null(res)) return(NULL)
  res$result
}

ora_up   <- run_ora_gprofiler(sig_up,   background)
ora_down <- run_ora_gprofiler(sig_down, background)

n_terms <- function(x) if (is.null(x)) 0 else nrow(x)
cat("Up   — significant terms:", n_terms(ora_up),   "\n")
cat("Down — significant terms:", n_terms(ora_down), "\n")

<div style="background:#d1ecf1;border-left:4px solid #0c5460;padding:10px;margin:10px 0;">

  <strong>Note:</strong> If a direction returns <code>NULL</code> (no query genes, or no significant
  terms), the tables and plots below will simply be empty for that direction. This is a valid outcome,
  not an error.

</div>

### Inspect Significant Terms

**Up-regulated genes — enriched terms**

In [ ]:
if (!is.null(ora_up) && nrow(ora_up) > 0) {
  DT::datatable(
    ora_up %>%
      select(source, term_id, term_name, term_size,
             query_size, intersection_size, p_value) %>%
      arrange(p_value) %>%
      mutate(p_value = signif(p_value, 4)),
    rownames   = FALSE,
    colnames   = c("Source", "Term ID", "Term", "Term size",
                   "Query size", "Hits", "p-value (g:SCS)"),
    extensions = c("Buttons", "Scroller"),
    options    = list(dom = "Bfrtip", buttons = c("copy", "csv"),
                      scrollX = TRUE, scrollY = 300, scroller = TRUE),
    caption = "Significant ORA terms — UP-regulated genes"
  )
} else {
  cat("No significant enriched terms for up-regulated genes.\n")
}

**Down-regulated genes — enriched terms**

In [ ]:
if (!is.null(ora_down) && nrow(ora_down) > 0) {
  DT::datatable(
    ora_down %>%
      select(source, term_id, term_name, term_size,
             query_size, intersection_size, p_value) %>%
      arrange(p_value) %>%
      mutate(p_value = signif(p_value, 4)),
    rownames   = FALSE,
    colnames   = c("Source", "Term ID", "Term", "Term size",
                   "Query size", "Hits", "p-value (g:SCS)"),
    extensions = c("Buttons", "Scroller"),
    options    = list(dom = "Bfrtip", buttons = c("copy", "csv"),
                      scrollX = TRUE, scrollY = 300, scroller = TRUE),
    caption = "Significant ORA terms — DOWN-regulated genes"
  )
} else {
  cat("No significant enriched terms for down-regulated genes.\n")
}

### Interactive Manhattan Plot

g:Profiler's native `gostplot()` shows all tested terms grouped by source, with significant terms highlighted. We run it once on the combined query for a compact overview.

In [ ]:
combined_query <- list(
  up   = sig_up,
  down = sig_down
)
# Drop empty directions to avoid an error
combined_query <- combined_query[lengths(combined_query) > 0]

if (length(combined_query) > 0) {
  gost_multi <- gost(
    query             = combined_query,
    organism          = "hsapiens",
    significant       = TRUE,
    user_threshold    = 0.05,
    correction_method = "g_SCS",
    domain_scope      = "custom",
    custom_bg         = background,
    sources           = c("GO:BP", "KEGG", "REAC")
  )
  if (!is.null(gost_multi)) {
    gostplot(gost_multi, capped = TRUE, interactive = TRUE)
  }
}

### Bar Plot — Top Terms by Direction

A static bar plot of the strongest terms in each direction, ranked by `-log10(p_value)`. The number of DE genes hitting each term is printed at the end of each bar.

In [ ]:
prep_ora <- function(x, label, n = 10) {
  if (is.null(x) || nrow(x) == 0) return(NULL)
  x %>%
    arrange(p_value) %>%
    slice_head(n = n) %>%
    transmute(
      term_name,
      hits        = intersection_size,
      neglog10p   = -log10(p_value),
      direction   = label
    )
}

ora_combined <- bind_rows(
  prep_ora(ora_up,   "Up-regulated"),
  prep_ora(ora_down, "Down-regulated")
)

if (!is.null(ora_combined) && nrow(ora_combined) > 0) {
  ora_combined <- ora_combined %>%
    mutate(
      direction  = factor(direction, levels = c("Up-regulated", "Down-regulated")),
      term_name  = reorder(paste(term_name, direction, sep = "___"), neglog10p)
    )

  ora_barplot <- ggplot(ora_combined,
                        aes(x = neglog10p, y = term_name, fill = direction)) +
    geom_col(width = 0.7, show.legend = FALSE) +
    geom_text(aes(label = hits), hjust = -0.2, size = 3.4, fontface = "bold") +
    geom_vline(xintercept = -log10(0.05), linetype = "dashed",
               color = "black", linewidth = 0.5) +
    facet_grid(direction ~ ., scales = "free_y", space = "free_y", switch = "y") +
    scale_y_discrete(labels = function(x) sub("___.*$", "", x)) +
    scale_fill_manual(values = c("Up-regulated"   = "firebrick",
                                 "Down-regulated" = "steelblue")) +
    scale_x_continuous(expand = expansion(mult = c(0, 0.16))) +
    theme_bw() +
    theme(
      strip.background   = element_blank(),
      strip.placement    = "outside",
      strip.text.y.left  = element_text(angle = 0, face = "bold", size = 11),
      panel.grid.major.y = element_blank()
    ) +
    labs(
      title    = "ORA — Top Enriched Terms by Direction (g:Profiler)",
      subtitle = "Dexamethasone vs Untreated | Human ASM",
      x        = "-log10(p-value, g:SCS)",
      y        = NULL
    )

  ora_barplot
} else {
  cat("No enriched terms to plot.\n")
}

## Part 2 — Gene Set Enrichment Analysis (GSEA)

**What is GSEA?**

**GSEA asks: *"Are genes in a gene set coordinately shifting up or down across my full ranked list?"***

Unlike ORA, GSEA:

- Uses **all tested genes** — capturing coordinated but subtle changes
- **Ranks genes** by a metric encoding both significance and direction
- Returns a **Normalised Enrichment Score (NES)**: positive = up in treatment; negative = down

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

 **Why signed log p-value for ranking?**

 `rank = sign(LFC) x -log10(pvalue)`

 Genes at the top are significantly upregulated; genes at the bottom are significantly
 downregulated. This is more robust than ranking on LFC alone, which can place noisy
 high-fold-change genes from lowly expressed genes at the extremes.

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

<strong>📘 Note:</strong> Genes with `pvalue = 0` (below machine precision in DESeq2) are excluded
 because `-log10(0) = Inf` is not a valid ranking score.

</div>

</div>

### Prepare the Ranked Gene List

In [ ]:
ranked_df <- res_df %>%
  filter(!is.na(pvalue), !is.na(log2FoldChange)) %>%
  filter(pvalue > 0) %>%
  mutate(rank_metric = sign(log2FoldChange) * -log10(pvalue)) %>%
  arrange(desc(rank_metric)) %>%
  distinct(gene, .keep_all = TRUE)

rank_vector        <- ranked_df$rank_metric
names(rank_vector) <- ranked_df$gene

cat("Genes in ranked list :", length(rank_vector), "\n")
cat("Infinite values      :", sum(!is.finite(rank_vector)), "\n")
cat("Top 5 (upregulated)  :", paste(head(names(rank_vector), 5), collapse = ", "), "\n")
cat("Bottom 5 (downreg.)  :", paste(tail(names(rank_vector), 5), collapse = ", "), "\n")

### Retrieve MSigDB Gene Sets

We use the MSigDB **Reactome** collection (`C2: CP:REACTOME`) for human, reshaped into the named list
`fgsea` expects. Reactome is a good fit here: it is a large, curated pathway collection that includes
detailed immune, inflammatory and signalling pathways — the biology this experiment is about.

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px;margin:10px 0;">

  <strong>Tip:</strong> To use a different collection, change the arguments below (e.g. Hallmark via
  <code>collection = "H"</code>, or GO:BP via <code>C5</code> / <code>GO:BP</code>). The subcollection
  and argument names for <code>msigdbr()</code> changed between versions, so the call below detects the
  right names automatically.

</div>

In [ ]:
# msigdbr argument names differ by version (collection/subcollection vs category/subcategory),
# and the KEGG subcollection was renamed, so we detect the available names first.
get_sets <- function(collection, subcollection = NULL) {
  df <- tryCatch(
    if (is.null(subcollection)) msigdbr(species = "Homo sapiens", collection = collection)
    else msigdbr(species = "Homo sapiens", collection = collection, subcollection = subcollection),
    error = function(e)
      if (is.null(subcollection)) msigdbr(species = "Homo sapiens", category = collection)
      else msigdbr(species = "Homo sapiens", category = collection, subcategory = subcollection)
  )
  split(x = df$gene_symbol, f = df$gs_name)
}

pathways_list <- get_sets("C2", "CP:REACTOME")

cat("Gene sets retrieved:", length(pathways_list), "\n")
cat("Example set        :", names(pathways_list)[1], "\n")
cat("First 5 genes      :", paste(head(pathways_list[[1]], 5), collapse = ", "), "\n")

### Run GSEA

We use `fgseaMultilevel`, which resolves very small p-values more accurately than the simple-permutation
`fgsea()`, with `minSize = 15` and `maxSize = 1000` to skip tiny unstable sets and enormous, uninformative
ones.

In [ ]:
set.seed(42)

gsea_results <- fgseaMultilevel(
  pathways = pathways_list,
  stats    = rank_vector,
  minSize  = 15,
  maxSize  = 1000
)

cat("Gene sets tested         :", nrow(gsea_results), "\n")
cat("Significant (padj < 0.05):", sum(gsea_results$padj < 0.05, na.rm = TRUE), "\n")
cat("Nominal (pval < 0.05)    :", sum(gsea_results$pval < 0.05, na.rm = TRUE), "\n")

<div style="background:#eaf4fd;border-left:5px solid #3498db;padding:0.5em 1em;margin:1em 0;border-radius:6px;">

**What we see, and why it differs from ORA.** In this dataset GSEA finds **no gene set significant after
multiple-testing correction** (padj < 0.05), even though ORA found strongly enriched cytokine and immune
pathways. This is not a contradiction — the two methods ask different questions:

- **ORA** asks whether the ~700 *significant* DE genes are concentrated in a pathway. They are (in cytokine
  and immune pathways), so ORA lights up.
- **GSEA** asks whether an *entire* gene set shifts coherently across all 20,000 genes. The dexamethasone
  response here is carried by a few hundred strongly-changed genes scattered across many pathways, rather
  than whole pathways moving together — so no set reaches the coordinated shift GSEA rewards.

This is a real biological result, not a failure: **the glucocorticoid effect is gene-specific, not
pathway-coherent**, which is exactly the situation where ORA is the more sensitive tool. The nominal
(uncorrected) top sets below still point at the right biology — immune and inflammatory pathways — showing
GSEA *sees* the theme but cannot call it significant on this sample size.

</div>

### Inspect Gene Sets

In [ ]:
# Sets passing multiple-testing correction (expected: none in this dataset)
gsea_padj_sig <- gsea_results %>% as.data.frame() %>% filter(padj < 0.05)

# Sets passing the nominal (uncorrected) threshold — for inspecting the biology
gsea_nom <- gsea_results %>%
  as.data.frame() %>%
  filter(pval < 0.05) %>%
  arrange(pval)

cat("Sets significant after correction (padj < 0.05):", nrow(gsea_padj_sig), "\n")
cat("Sets at nominal pval < 0.05                    :", nrow(gsea_nom), "\n")

DT::datatable(
  gsea_nom %>%
    select(pathway, NES, pval, padj, size) %>%
    mutate(
      direction = ifelse(NES > 0, "Activated", "Suppressed"),
      across(where(is.numeric), \(x) signif(x, 3))
    ),
  rownames   = FALSE,
  colnames   = c("Gene set", "NES", "p-value", "padj", "Size", "Direction"),
  extensions = c("Buttons", "Scroller"),
  options    = list(dom = "Bfrtip", buttons = c("copy", "csv"),
                    scrollX = TRUE, scrollY = 300, scroller = TRUE),
  caption = "GSEA gene sets at nominal p < 0.05 (none survive padj correction — see note above)"
)

### NES Bar Plot — Top Sets by Nominal p-value

Because no set survives correction, we plot the top sets by **nominal** p-value. These do not represent
statistically confirmed enrichment; they show the *direction* of the biology GSEA detects. Read them as
suggestive, not conclusive.

In [ ]:
gsea_bar <- gsea_nom %>%
  slice_min(pval, n = 20) %>%
  mutate(
    direction = ifelse(NES > 0, "Activated", "Suppressed"),
    pathway   = fct_reorder(pathway, NES)
  )

if (nrow(gsea_bar) > 0) {
  gsea_nesplot <- ggplot(gsea_bar,
                         aes(x = NES, y = pathway, fill = direction)) +
    geom_col(width = 0.7) +
    geom_vline(xintercept = 0, color = "black", linewidth = 0.5) +
    theme_bw() +
    theme(legend.position = "bottom") +
    labs(
      title    = "GSEA — top sets by nominal p-value (not padj-significant)",
      subtitle = "Dexamethasone vs Untreated | Human ASM (Reactome)",
      x        = "Normalised Enrichment Score (NES)",
      y        = NULL,
      fill     = NULL
    )

  gsea_nesplot
} else {
  cat("No gene sets with nominal pval < 0.05 to plot.\n")
}

### Dot Plot — NES, p-value and Gene Set Size

In [ ]:
if (nrow(gsea_bar) > 0) {
  gsea_dotplot <- ggplot(gsea_bar,
                         aes(x = NES, y = pathway, size = size, color = pval)) +
    geom_point(alpha = 0.85) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
    scale_size_continuous(name = "Genes in set", range = c(3, 10)) +
    theme_bw() +
    theme(legend.position = "right") +
    labs(
      title    = "GSEA — Gene Set Dot Plot (nominal p)",
      subtitle = "Point size = genes in set | Colour = nominal p-value",
      x        = "NES",
      y        = NULL
    )

  gsea_dotplot
}

### Enrichment Plot — Top Sets

The enrichment plot shows the running sum of the enrichment score for a single gene set across the ranked gene list. The peak of the curve is the enrichment score; genes from the set are shown as vertical bars (the "rug") at the bottom.

In [ ]:
top_up <- gsea_nom %>% filter(NES > 0) %>% slice_min(pval, n = 1) %>% pull(pathway)
top_down <- gsea_nom %>% filter(NES < 0) %>% slice_min(pval, n = 1) %>% pull(pathway)

if (length(top_up) > 0) {
  print(
    plotEnrichment(pathways_list[[top_up]], rank_vector) +
      labs(title = paste("Top activated (nominal):", top_up))
  )
}

if (length(top_down) > 0) {
  print(
    plotEnrichment(pathways_list[[top_down]], rank_vector) +
      labs(title = paste("Top suppressed (nominal):", top_down))
  )
}

## Part 3 — Comparing ORA and GSEA

ORA and GSEA answer different questions, and in this dataset they give very different verdicts:
**ORA finds strongly enriched cytokine and immune pathways, while GSEA finds no set that survives
multiple-testing correction.** This section makes that contrast explicit, because it is one of the most
useful things to understand about enrichment analysis.

In [ ]:
ora_terms <- character(0)
if (!is.null(ora_up))   ora_terms <- c(ora_terms, ora_up$term_name)
if (!is.null(ora_down)) ora_terms <- c(ora_terms, ora_down$term_name)
ora_terms <- unique(ora_terms)

gsea_padj_terms <- gsea_results %>% as.data.frame() %>% filter(padj < 0.05) %>% pull(pathway)
gsea_nom_terms  <- gsea_nom$pathway

cat("ORA significant terms (padj-corrected)   :", length(ora_terms),        "\n")
cat("GSEA sets significant after correction   :", length(gsea_padj_terms),  "\n")
cat("GSEA sets at nominal p < 0.05 (suggestive):", length(gsea_nom_terms),  "\n")

<div style="background:#d4edda;border-left:4px solid #28a745;padding:0.5em 1em;margin:1em 0;border-radius:6px;">

**The teaching point — why the two methods disagree here.**

- **ORA** tests whether the ~700 genes that individually passed significance are *concentrated* in a
  pathway. They are — in cytokine and immune pathways — so ORA reports strong, corrected-significant
  enrichment that matches the paper's biology.
- **GSEA** tests whether an *entire* gene set moves *coherently* across the full ranked list. The
  dexamethasone response is carried by a few hundred strongly-changed genes spread across many pathways,
  not by whole pathways shifting together, so no set reaches significance after correction — even though
  the nominal top hits (immune, inflammatory, chemokine sets) point at the right biology.

**Bad results are results.** The GSEA "non-finding" is informative: it tells us the glucocorticoid effect
here is **gene-specific rather than pathway-coordinated**. When the biology looks like this, ORA is the
more appropriate tool — which is exactly why we report ORA as the primary enrichment result and GSEA as a
secondary, directional check.

</div>

## Saving Results

In [ ]:
results_dir <- file.path(git_root, "results", "human")
dir.create(results_dir, showWarnings = FALSE)

ora_all <- bind_rows(
  if (!is.null(ora_up))   ora_up   %>% mutate(direction = "Up-regulated"),
  if (!is.null(ora_down)) ora_down %>% mutate(direction = "Down-regulated")
)

if (!is.null(ora_all) && nrow(ora_all) > 0) {
  # gost result can contain list-columns (e.g. parents); drop them for TSV export
  ora_all_flat <- ora_all %>% select(where(~ !is.list(.)))
  write.table(
    ora_all_flat,
    file      = file.path(results_dir, "ORA_gProfiler_treatment_vs_control.tsv"),
    sep       = "\t", quote = FALSE, row.names = FALSE
  )
}

write.table(
  as.data.frame(gsea_results) %>% select(-leadingEdge),
  file      = file.path(results_dir, "GSEA_Reactome_treatment_vs_control.tsv"),
  sep       = "\t", quote = FALSE, row.names = FALSE
)

cat("Results saved to:", results_dir, "\n")
cat("  ORA  -> ORA_gProfiler_treatment_vs_control.tsv\n")
cat("  GSEA -> GSEA_Reactome_treatment_vs_control.tsv\n")

## Summary

| Step | Choice made | Rationale |
|---|---|---|
| ORA tool | `gprofiler2::gost()` | Native human support; GO/KEGG/Reactome in one call; no ID mapping needed |
| ORA background | All DESeq2-tested genes (`custom_bg`, `domain_scope = "custom"`) | Correct statistical background — not the whole genome |
| ORA correction | g:SCS | g:Profiler's default; accounts for term overlap structure |
| GSEA tool | `fgseaMultilevel` | Accurate small p-values; proper NES and permutation-based testing |
| GSEA gene sets | MSigDB Reactome via `msigdbr` | Curated, detailed pathways; where the immune/inflammatory themes surfaced |
| GSEA rank metric | sign(LFC) x -log10(pvalue) | Encodes direction and significance; robust to lowly expressed gene noise |
| Primary result | ORA (g:Profiler) | Cytokine/immune enrichment, corrected-significant, matches the paper |
| GSEA outcome | No padj-significant sets | Effect is gene-specific, not pathway-coherent; nominal top sets still immune/inflammatory |

Enrichment results are saved and ready for biological interpretation and reporting. The primary,
statistically-supported finding is the ORA enrichment of cytokine and immune pathways, consistent with the
glucocorticoid biology described by Himes *et al.* GSEA is reported as a secondary, directional check.

</br>

In [ ]:
sessionInfo()